In [ ]:
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
URLS = [
    {
        "url": "https://en.wikivoyage.org/wiki/Berlin",
        "city": "berlin",
        "category": "sightseeing",
        "price_level": "medium"
    },
    {
        "url": "https://en.wikipedia.org/wiki/Berlin",
        "city": "berlin",
        "category": "art",
        "price_level": "medium"
    },
    {
        "url": "https://en.wikivoyage.org/wiki/Berlin",
        "city": "berlin",
        "category": "food",
        "price_level": "cheap"
    }
]

# Added USER-AGENT header here becoz previously the RAG pipeline was getting blocked
def fetch_text(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
        }

        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            print(f"Failed to fetch {url} - Status code: {response.status_code}")
            return ""

        soup = BeautifulSoup(response.text, "html.parser")

        # remove junk
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator="\n")
        text = "\n".join([line.strip() for line in text.splitlines() if line.strip()])
        return text

    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return ""

In [ ]:
sample_text = fetch_text(URLS[0]["url"])
print(sample_text[:1000])  # print first 1000 characters

In [ ]:
#CHUNKING
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

In [ ]:
#Testing Chunking
chunks = chunk_text(sample_text)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

In [ ]:
#Attaching Metdata 
def create_documents():
    all_data = []

    for item in URLS:
        text = fetch_text(item["url"])
        chunks = chunk_text(text)

        for chunk in chunks:
            all_data.append({
                "text": chunk,
                "url": item["url"],
                "city": item["city"],
                "category": item["category"],
                "price_level": item["price_level"]
            })

    return all_data

In [ ]:
#Generating Final dataaset
data = create_documents()

print("Total chunks:", len(data))
print("\nSample document:\n")
print(data[0])

In [ ]:
# ************** PREFERANCE Extraction ***************
!pip install sentence-transformers faiss-cpu

In [ ]:
import os
from groq import Groq
import json

# Initialize Groq client (ensure your API key is in environment variables)
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def extract_preferences(user_query):
    system_prompt = """
    You are a travel assistant's parsing engine. 
    Extract the following from the user query:
    1. city (string)
    2. price_level (must be one of: cheap, medium, expensive)
    3. category (must be a list containing: food, art, or sightseeing)

    Return ONLY a valid JSON object. Do not include any conversational text.
    If a value is missing, use null.
    """
    
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        model="llama-3.3-70b-versatile", # Or your preferred Groq model [cite: 8]
        response_format={"type": "json_object"} # Forces JSON output
    )
    
    return json.loads(chat_completion.choices[0].message.content)

query = "I want a 3-day Berlin trip with cheap food and some art"
prefs = extract_preferences(query)
print(prefs)

In [ ]:
# # Retrieval + re-ranking ( this is to connect search query to vector datavase using user's preferrrences )
# # I will use Facebook AI Similarity Search vector DB FAISS
# from sentence_transformers import SentenceTransformer
# import numpy as np
# # Choice: all-MiniLM-L6-v2 (Fast, lightweight, and perfect for travel queries)
# model = SentenceTransformer('all-MiniLM-L6-v2')

# def search_travel_data(query_text, extracted_prefs, vector_db_index, metadata_list, top_k=10):
#     # Convert query to vector
#     query_vector = model.encode([query_text])
    
#     # Search the Vector DB for the top 10 most similar chunks
#     distances, indices = vector_db_index.search(np.array(query_vector), top_k)
    
#     results = []
#     for idx in indices[0]:
#         chunk_metadata = metadata_list[idx]
        
#         # Check if the city matches the user's extracted preference
#         if chunk_metadata['city'].lower() == extracted_prefs['city'].lower():
#             results.append({
#                 "content": chunk_metadata['text'],
#                 "url": chunk_metadata['url'],
#                 "score": 0 # Placeholder for re-ranking
#             })
            
#     return results

In [ ]:
pip install fastembed faiss-cpu
pip install faiss-cpu scikit-learn numpy


In [9]:
URLS = [
    {
        "url": "https://en.wikivoyage.org/wiki/Berlin",
        "city": "berlin",
        "category": "sightseeing",
        "price_level": "medium"
    },
    {
        "url": "https://en.wikipedia.org/wiki/Berlin",
        "city": "berlin",
        "category": "art",
        "price_level": "medium"
    },
    {
        "url": "https://en.wikivoyage.org/wiki/Berlin",
        "city": "berlin",
        "category": "food",
        "price_level": "cheap"
    }
]

documents = [
    {
        "text": "Berlin is a top sightseeing destination with historical landmarks like Brandenburg Gate and museums.",
        **item
    }
    for item in URLS
]

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer()

texts = [d["text"] for d in documents]
embeddings = vectorizer.fit_transform(texts).toarray().astype("float32")

In [11]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(embeddings)

In [12]:
query = "3 day Berlin trip with cheap food and art"

query_vec = vectorizer.transform([query]).toarray().astype("float32")

D, I = index.search(query_vec, k=3)

retrieved_docs = [documents[i] for i in I[0]]

In [13]:
def filter_docs(docs, prefs):
    filtered = []

    for d in docs:
        if prefs.get("city") and d["city"] != prefs["city"]:
            continue

        if prefs.get("category") and d["category"] not in prefs["category"]:
            continue

        if prefs.get("price_level") and d["price_level"] != prefs["price_level"]:
            continue

        filtered.append(d)

    return filtered

def rerank(query, docs):
    query_words = set(query.lower().split())

    def score(doc):
        score = 0

        # keyword overlap
        doc_words = set(doc["text"].lower().split())
        score += len(query_words.intersection(doc_words))

        # boost cheap options (travel use case logic)
        if doc["price_level"] == "cheap":
            score += 2

        # boost relevant categories
        if doc["category"] in ["food", "art"]:
            score += 2

        return score

    return sorted(docs, key=score, reverse=True)

def retrieval_pipeline(query, preferences):
    query_vec = vectorizer.transform([query]).toarray().astype("float32")
    D, I = index.search(query_vec, k=3)

    retrieved = [documents[i] for i in I[0]]

    filtered = filter_docs(retrieved, preferences)
    reranked = rerank(query, filtered)

    return reranked

In [14]:
preferences = {
    "city": "berlin",
    "category": ["food", "art"],
    "price_level": "cheap"
}

query = "3 day Berlin trip with cheap food and art"

results = retrieval_pipeline(query, preferences)

for r in results:
    print(r)

{'text': 'Berlin is a top sightseeing destination with historical landmarks like Brandenburg Gate and museums.', 'url': 'https://en.wikivoyage.org/wiki/Berlin', 'city': 'berlin', 'category': 'food', 'price_level': 'cheap'}


In [15]:
def context_quality_check(chunks):
    """
    Returns: context_good OR context_insufficient
    """

    if len(chunks) == 0:
        return "context_insufficient"

    # simple heuristic: check if we actually have useful info
    total_text = " ".join([c["text"] for c in chunks]).lower()

    if len(total_text.split()) < 10:
        return "context_insufficient"

    return "context_good"

In [16]:
def get_final_context(query, preferences):
    chunks = retrieval_pipeline(query, preferences)

    status = context_quality_check(chunks)

    if status == "context_good":
        return chunks

    # fallback: relax filters
    relaxed_prefs = {"city": preferences.get("city")}

    chunks = retrieval_pipeline(query, relaxed_prefs)

    return chunks

In [17]:
!pip install groq

In [18]:
from groq import Groq

client = Groq(api_key="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")

In [19]:
def generate_answer(query, chunks):

    context_text = "\n".join(
        [f"- {c['text']} (source: {c['url']})" for c in chunks]
    )

    prompt = f"""
You are a travel assistant.

Use ONLY the context below to answer.

Context:
{context_text}

User Query:
{query}

Give a helpful travel plan. Keep it grounded in the context.
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    return response.choices[0].message.content

In [20]:
print("=== RAG TRAVEL ASSISTANT ===")

query = input("Enter your travel query: ")

preferences = {
    "city": "berlin",
    "category": ["food", "art"],
    "price_level": "cheap"
}

chunks = get_final_context(query, preferences)
answer = generate_answer(query, chunks)

print("\n===== FINAL ANSWER =====\n")
print(answer)

print("\n===== DEBUG INFO =====")
print("Preferences:", preferences)

print("\nRetrieved Chunks:")
for c in chunks:
    print("-", c["text"], "|", c["url"])

=== RAG TRAVEL ASSISTANT ===


Enter your travel query:  3 day trip plan



===== FINAL ANSWER =====

Berlin is a fantastic destination for a 3-day trip. Here's a suggested itinerary for you:

**Day 1: Explore Berlin's Iconic Landmarks**

- Morning: Start your day at the iconic Brandenburg Gate (Brandenburger Tor), one of Berlin's most recognizable landmarks. Take a stroll through the adjacent Tiergarten park.
- Afternoon: Visit the nearby Reichstag building, home to the German parliament. You can take a guided tour of the dome for a panoramic view of the city.
- Evening: Head to the trendy Kreuzberg neighborhood for dinner and explore the vibrant nightlife.

**Day 2: Discover Berlin's Rich History and Culture**

- Morning: Visit the German History Museum (Deutsches Historisches Museum) to learn about Germany's complex past.
- Afternoon: Explore the Museum Island (Museumsinsel), a UNESCO World Heritage site, which houses five of Berlin's most important museums, including the Alte Nationalgalerie and the Pergamon Museum.
- Evening: Enjoy a traditional German d